# Visualization of Results

This notebook visualizes:
- Whole-brain correlation matrices
- Network-level correlation patterns


In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from src.visualization import plot_correlation_matrix_transposed, plot_data_gender_networks, plot_nested_boxplot_gender_network
from src.statistics import slice_entropies, whole_brain

In [ ]:
whole_brain_cor = whole_brain()
plot_correlation_matrix_transposed(whole_brain_cor)

In [ ]:
sliced_brain = slice_entropies()

flat_data = [(key1, key2, key3, key4, value4[0][0], value4[1][0], value4[2][0]) 
             for key1, value1 in sliced_brain.items() 
             for key2, value2 in value1.items() 
             for key3, value3 in value2.items()
             for key4, value4 in value3.items()]

df = pd.DataFrame(flat_data, columns=['Group', 'Variable', 'Statistic', 'Network', 'Correlation', 'P-Value_Uncorrected', 'P-Value_Corrected'])

statistic_mapping = {
    'roc_raw_squared': 'MSSD',
    'std_raw': 'SD',
    'se': 'SE',
    'fe': 'FE',
    'mse': 'MSE'
}

task_markers = {
    'rest1_s200': 'o', 'rest2_s200': 's', 'emotion_s200': '^', 'gambling_s200': 'D', 'language_s200': 'v', 
    'motor_s200': 'p', 'relational_s200': 'X', 'social_s200': '*', 'wm_s200': '<'
}

task_mapping = {
    'rest1_s200': 'REST1',
    'rest2_s200': 'REST2',
    'emotion_s200': 'EMOTION',
    'gambling_s200': 'GAMBLING',
    'language_s200': 'LANGUAGE',
    'motor_s200': 'MOTOR',
    'relational_s200': 'RELATIONAL',
    'social_s200': 'SOCIAL',
    'wm_s200': 'WM'
}

# Change here for different Entropy or Variability
rest1_df = df[(df['Statistic'] == 'se')]
#|(df['Statistic'] == 'mse') | (df['Statistic'] == 'fe')

def map_task(task):
    for key in task_markers:
        if key.lower() in task.lower():  
            return key
    return task  

sns.set_style("whitegrid")

for group, group_data in rest1_df.groupby('Group'):
    plt.figure(figsize=(10, 6))  
    
    handles = []
    labels = []
    label_added = {}

    for task, task_data in group_data.groupby('Variable'):
        
        if task not in task_markers:
            print(f"Warning: Task '{task}' not found in task_markers! Skipping this task.")
            continue  
        
        for network, network_data in task_data.groupby('Network'):
            
            statistic_palette = sns.color_palette("husl", len(network_data['Statistic'].unique()))
            statistic_colors = {stat: color for stat, color in zip(network_data['Statistic'].unique(), statistic_palette)}
            
            for statistic, statistic_data in network_data.groupby('Statistic'):
                
                marker = task_markers[task]  
                
                significant_data = statistic_data[statistic_data['P-Value_Corrected'] < 0.05]
                non_significant_data = statistic_data[statistic_data['P-Value_Corrected'] >= 0.05]
                
                scatter_sig = plt.scatter(
                    significant_data['Network'], 
                    significant_data['Correlation'], 
                    label=None, 
                    color=statistic_colors[statistic], 
                    marker=marker, 
                    alpha=1, 
                    s=80
                )
                
                scatter_non_sig = plt.scatter(
                    non_significant_data['Network'], 
                    non_significant_data['Correlation'], 
                    label=None, 
                    color=statistic_colors[statistic], 
                    marker=marker, 
                    alpha=0.2, 
                    s=80
                )
                
                label = f"{task_mapping.get(task, task)} - {statistic_mapping.get(statistic, statistic)}"
                
                if label not in label_added:
                    legend_handle = plt.Line2D([], [], color=statistic_colors[statistic], marker=marker,
                                               linestyle='None', markersize=10, label=label)
                    handles.append(legend_handle)
                    labels.append(label)
                    label_added[label] = True

    plt.xlabel('Network')
    plt.ylabel('Correlation')
    plt.title(f"{group}")
    
    legend = plt.legend(handles, labels, loc='upper left', bbox_to_anchor=(1, 1), ncol=2)
    
    plt.tight_layout()
    plt.show()

# Plot for Gender

In [ ]:
gender_plot = plot_data_gender_networks() # see function for usage instruction - you have to say in the function what you want to plot

plot_nested_boxplot_gender_network(gender_plot, 'Gender') # same here